# Нейросети I: многослойный перцептрон (MLP)

## Содержание

0. Подготовка среды и подключение Weights & Biases
1. Данные: вернёмся к оттоку клиентов
2. Нейрон и слой: математический аппарат
3. Первая нейросеть на PyTorch
4. Обучение: эпохи и функция потерь
5. Оценка и сравнение с классическими моделями
6. Практикум

Это первое занятие про нейросети. Мы не будем сразу строить что-то сложное: разберём базовый «кирпичик» — **нейрон и слой**, соберём маленькую полносвязную сеть (**MLP**), обучим её и сравним с логистической регрессией из занятия 3. Идея в том, чтобы увидеть, что нейросеть — это знакомые вам вещи (линейная комбинация + нелинейность + градиентный спуск), просто собранные вместе.

Демонстрационный пример — тот же прогноз оттока клиентов, что и в занятии 3. Практикум (раздел 6) вы выполняете на **датасете своего проекта**. Выводы после каждого блока записывайте **в ячейке markdown прямо в блокноте**.

## 0. Подготовка среды и подключение Weights & Biases

Для этого занятия нужен **PyTorch**. В Google Colab он уже установлен. Локально: `pip install torch`. Это занятие можно целиком выполнять на CPU — сеть маленькая.

**Примечание.** Если с W&B проблемы, замените `wandb.init(...)` на `wandb.init(mode="disabled")`.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

import wandb

torch.manual_seed(42)
RANDOM_STATE = 42
plt.rcParams['figure.figsize'] = (8, 5)
wandb.login()

ModuleNotFoundError: No module named 'wandb'

## 1. Данные: вернёмся к оттоку клиентов

Берём тот же датасет, что и в занятии 3 (Telco Customer Churn). Подготовка такая же, как раньше: пропуски в числовых признаках заполняем медианой, числовые стандартизуем, категориальные кодируем one-hot. Весь препроцессинг — внутри `Pipeline`, чтобы не было утечки.

**Важно для нейросетей:** они работают с числами и чувствительны к масштабу, поэтому предобработка не опциональна, а обязательна.

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)

d = df.drop(columns=['customerID']).copy()
d['TotalCharges'] = pd.to_numeric(d['TotalCharges'], errors='coerce')
d['Churn'] = (d['Churn'] == 'Yes').astype(int)

num = [c for c in d.select_dtypes(include=np.number).columns if c != 'Churn']
cat = [c for c in d.columns if c not in num + ['Churn']]
X, y = d[num + cat], d['Churn'].to_numpy()

# 60/20/20 со стратификацией
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=RANDOM_STATE)

prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num),
    ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20), cat),
])


def dense(M):
    return (M.toarray() if hasattr(M, 'toarray') else np.asarray(M)).astype('float32')


Xtr = dense(prep.fit_transform(X_train))
Xval = dense(prep.transform(X_val))
Xte = dense(prep.transform(X_test))
print("Обучающая:", Xtr.shape, "| признаки:", Xtr.shape[1])

In [ ]:
# Подаём данные в нейросеть батчами: TensorDataset + DataLoader


def to_dataset(X, y):
    return TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.float32).unsqueeze(1))


train_loader = DataLoader(to_dataset(Xtr, y_train), batch_size=256, shuffle=True)
Xval_t = torch.tensor(Xval)
Xte_t = torch.tensor(Xte)
print("Батчей в эпохе:", len(train_loader))

## 2. Нейрон и слой: математический аппарат

**Нейрон** — это линейная комбинация входов, пропущенная через нелинейную функцию активации:

$$z = \sum_i w_i x_i + b, \qquad a = f(z)$$

Функция активации делает модель нелинейной. Чаще всего берут **ReLU**: $f(z) = \max(0, z)$. На выходе для бинарной классификации — **сигмоида** $\sigma(z) = \frac{1}{1+e^{-z}}$, и результат читается как вероятность (как в логистической регрессии из занятия 3).

**Слой** — это сразу много нейронов. Удобно записать матрично: $a = f(Wx + b)$, где $W$ — матрица весов, $b$ — вектор смещений. **MLP (многослойный перцептрон)** — это несколько таких слоёв подряд: выход одного становится входом следующего.

**Обучение** ничем не отличается от того, что было раньше:

- функция потерь для бинарной классификации — **бинарная кросс-энтропия** (та же, что в занятии 3);
- веса подбираются **градиентным спуском**, но не по всем данным сразу, а по **мини-батчам**;
- производные по весам считает **алгоритм обратного распространения (backprop)** — это правило цепочки, которое вы проходили в курсе 1. В PyTorch его считает `loss.backward()`, вручную писать не нужно.

**Задача 1.**
Покажите, что если убрать функции активации, то два линейных слоя подряд эквивалентны одному линейному слою. Почему без нелинейности нейросеть не может моделировать ничего, кроме линейной зависимости?

**Ваш вывод:**

_..._

## 3. Первая нейросеть на PyTorch

Соберём простую сеть: один входной слой, один скрытый слой на 16 нейронов с ReLU и один выходной нейрон. Для бинарной классификации удобно использовать `BCEWithLogitsLoss` — она объединяет сигмоиду и кросс-энтропию и численно устойчива.

In [ ]:
n_features = Xtr.shape[1]

model = nn.Sequential(
    nn.Linear(n_features, 16),   # скрытый слой: 16 нейронов
    nn.ReLU(),                   # нелинейность
    nn.Linear(16, 1),            # выход: один нейрон (логит)
)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print(model)

## 4. Обучение: эпохи и функция потерь

**Эпоха** — один полный проход по обучающим данным. Внутри эпохи идут мини-батчи: для каждого считается loss, берётся градиент и делается шаг оптимизатора. Будем сохранять средний loss по эпохам и следить за кривой обучения.

In [ ]:
epochs = 30
history = []

for epoch in range(epochs):
    model.train()
    total = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()          # обнуляем старые градиенты
        preds = model(xb)              # прямой проход
        loss = loss_fn(preds, yb)      # функция потерь
        loss.backward()                # backprop: считаем градиенты
        optimizer.step()               # шаг градиентного спуска
        total += loss.item() * len(xb)
    history.append(total / len(Xtr))
    if (epoch + 1) % 10 == 0:
        print(f"эпоха {epoch + 1}: train loss = {history[-1]:.3f}")

plt.plot(range(1, epochs + 1), history)
plt.xlabel('эпоха'); plt.ylabel('train loss')
plt.title('Кривая обучения')
plt.show()

**Примечание.** Loss убывает — сеть учится. Если кривая выходит на плато и не падает, попробуйте увеличить число эпох; если loss на train падает, а качество на validation нет — сеть переобучается (в следующем занятии разберём, как с этим бороться: dropout, ранняя остановка и т.д.).

## 5. Оценка и сравнение с классическими моделями

Получаем вероятности на тесте (не забудьте сигмоиду, потому что модель выдаёт логиты) и считаем метрики из занятия 3: **Average Precision** и **ROC-AUC**. Сравним с логистической регрессией на тех же данных.

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(Xte_t).numpy().ravel()
proba = 1 / (1 + np.exp(-logits))   # сигмоида

mlp_ap = average_precision_score(y_test, proba)
mlp_roc = roc_auc_score(y_test, proba)

lr = LogisticRegression(C=0.3, max_iter=1500, solver='liblinear').fit(Xtr, y_train)
lr_proba = lr.predict_proba(Xte)[:, 1]

print(f"MLP (нейросеть):  AP = {mlp_ap:.3f}, ROC-AUC = {mlp_roc:.3f}")
print(f"Логистическая регрессия: AP = {average_precision_score(y_test, lr_proba):.3f}, "
      f"ROC-AUC = {roc_auc_score(y_test, lr_proba):.3f}")

run = wandb.init(project="apml-course", name="mlp-churn", config={"hidden": 16, "epochs": epochs, "lr": 0.01})
wandb.log({"mlp_ap": mlp_ap, "mlp_roc_auc": mlp_roc})
run.finish()

**Задача 2.**
Поменяйте число нейронов в скрытом слое (например, 4, 16, 64) и число эпох. Как меняется качество и кривая обучения? Сравните нейросеть с логистической регрессией: заметен ли выигрыш и почему?

**Вывод:**

_..._

## 6. Практикум

Задания выполняются на **датасете своего проекта**. Нейросеть — ещё одна модель для вашей задачи классификации (или регрессии). После каждого задания заполните ячейку **«Вывод»**.

### Задание 6.1. Подготовка данных

Возьмите числовые признаки (категориальные закодируйте one-hot), обработайте пропуски и стандартизуйте. Разделите данные на train/val/test со стратификацией и переведите в тензоры и батчи. Сформулируйте целевую переменную.

**Вывод:**

_..._

### Задание 6.2. Обучение MLP

Соберите простой MLP (один-два скрытых слоя), задайте функцию потерь и оптимизатор. Обучите его и постройте кривую обучения. Подберите число эпох по качеству на validation.

**Вывод:**

_..._

### Задание 6.3. Оценка и сравнение

Посчитайте метрики на test (AP, ROC-AUC, а для регрессии — RMSE/R²) и сравните нейросеть с логистической регрессией или бустингом из предыдущих занятий. Залогируйте запуск в W&B.

**Вывод:**

_..._

### Задание 6.4. Итоговое резюме

В 5–7 предложениях: превзошла ли нейросеть классические модели на ваших данных; как размер сети и число эпох влияли на результат; что вы возьмёте в проект на зачёт.

**Резюме:**

_..._

---

## P.S. Зачем нейросети на табличных данных (и когда нет)

На небольших табличных данных **градиентный бустинг** из занятия 4 обычно не хуже, а часто лучше простого MLP — и работает без подбора архитектуры. Нейросети выигрывают, когда:

- данных много и есть сложные нелинейные зависимости;
- есть категориальные признаки с большой кардинальностью (эмбеддинги);
- задача про **изображения, текст или последовательности** — там работают CNN/LSTM, и это следующий шаг.

Именно поэтому на табличных задачах нейросеть разумно держать как «ещё один кандидат», который сравнивают с бустингом на одной валидации, а не как выбор по умолчанию.

### Справочные материалы

- PyTorch, основы (tutorial): https://pytorch.org/tutorials/beginner/basics/intro.html
- `nn.Module` и слои: https://pytorch.org/docs/stable/nn.html
- Функции потерь: https://pytorch.org/docs/stable/nn.html#loss-functions
- Оптимизаторы (Adam и др.): https://pytorch.org/docs/stable/optim.html